In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.saving import load_model
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)
from tqdm import tqdm
import pickle
import keras
import tensorflow as tf
from keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, concatenate, MultiHeadAttention, LayerNormalization, Add
from keras.models import Model
from sklearn.utils import shuffle

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom")
def f2_score(y_true, y_pred):
    # Ensure inputs are tensors
    y_true = tf.convert_to_tensor(y_true, dtype=tf.float32)
    y_pred = tf.convert_to_tensor(y_pred, dtype=tf.float32)
    # Calculate true positives, false positives, and false negatives
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1 - y_pred))
    # Calculate precision and recall
    epsilon = tf.keras.backend.epsilon()  # Small constant to avoid division by zero
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)

    # Calculate F2 score
    f2 = (5 * precision * recall) / (4 * precision + recall + epsilon)
    return f2.numpy()




@tf.keras.utils.register_keras_serializable(package="Custom")
class F2Score(tf.keras.metrics.Metric):
    def __init__(self, name='f2_score', threshold=0.5, **kwargs):
        super(F2Score, self).__init__(name=name, **kwargs)
        self.tp = self.add_weight(name='tp', initializer='zeros')
        self.fp = self.add_weight(name='fp', initializer='zeros')
        self.fn = self.add_weight(name='fn', initializer='zeros')
        self.epsilon = tf.keras.backend.epsilon()
        self.threshold = threshold

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.cast(y_pred > self.threshold, tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        self.tp.assign_add(tf.reduce_sum(y_true * y_pred))
        self.fp.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.fn.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))

    def result(self):
        precision = self.tp / (self.tp + self.fp + self.epsilon)
        recall = self.tp / (self.tp + self.fn + self.epsilon)
        f2 = (5 * precision * recall) / (4 * precision + recall + self.epsilon)
        return f2

    def reset_state(self, sample_weight=None):
        self.tp.assign(0.0)
        self.fp.assign(0.0)
        self.fn.assign(0.0)

    def get_config(self):
        config = super(F2Score, self).get_config()
        config.update({'name': self.name, 'threshold': self.threshold})
        return config

    @classmethod
    def from_config(cls, config):
        # Recreate the layer from its config
        return cls(**config)

In [ ]:
# @tf.keras.utils.register_keras_serializable(package="Custom")
# class DeepSet(tf.keras.Model):
#     def __init__(self, input_dim, hidden_dim, output_dim, **kwargs):
#         super(DeepSet, self).__init__(**kwargs)
        
#         self.input_dim = input_dim
#         self.hidden_dim = hidden_dim
#         self.output_dim = output_dim
#         # Element-wise transformation: phi network
#         self.phi = tf.keras.Sequential([
#             Dense(self.hidden_dim, activation='relu'),
#             Dense(self.hidden_dim, activation='relu')
#         ])
#         # Post-aggregation transformation: rho network
#         self.rho = tf.keras.Sequential([
#             Dense(self.hidden_dim, activation='relu'),
#             Dense(self.output_dim, activation='relu')
#         ])

    
#     def call(self, x):
#         # Apply phi to each ICD code embedding
#         transformed = self.phi(x)  # Shape: (batch_size, num_codes, output_dim)
#         # Aggregate using sum (or other aggregation functions)
#         aggregated = tf.reduce_sum(transformed, axis=1)  # Shape: (batch_size, output_dim)
#         # Apply rho to the aggregated representation
#         output = self.rho(aggregated)  # Shape: (batch_size, output_dim)
#         return output

#     def get_config(self):
#         # Return the parameters required for serialization
#         config = super(DeepSet, self).get_config()
#         config.update({
#             "input_dim": self.input_dim,
#             "hidden_dim": self.hidden_dim,
#             "output_dim": self.output_dim
#         })
#         return config

#     @classmethod
#     def from_config(cls, config):
#         # Recreate the layer from its config
#         return cls(**config)

In [ ]:
 
@tf.keras.utils.register_keras_serializable(package="Custom")
class DeepSet(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, output_dim, num_encode, num_decode, **kwargs):
        super(DeepSet, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_encode = num_encode
        self.num_decode = num_decode
        # # Element-wise transformation: phi network
        # self.phi = tf.keras.Sequential([
        #     Dense(self.hidden_dim, activation='relu')
        # ])
        # # Post-aggregation transformation: rho network
        # self.rho = tf.keras.Sequential([
        #     Dense(self.output_dim, activation='relu')
        # ])

        # Element-wise transformation: phi network
        self.phi = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu') for _ in range(self.num_encode)
        ])

        # Post-aggregation transformation: rho network
        self.rho = tf.keras.Sequential([
            Dense(self.hidden_dim, activation='relu') for _ in range(self.num_decode - 1)
        ] + [Dense(self.output_dim, activation='relu')])  # Last layer should output correct dimension

    def call(self, x):
        transformed = self.phi(x)  # (batch_size, num_codes, hidden_dim)
        aggregated = tf.reduce_sum(transformed, axis=1)  # (batch_size, hidden_dim)
        output = self.rho(aggregated)  # (batch_size, output_dim)
        return output

    def get_config(self):
        config = super(DeepSet, self).get_config()
        config.update({
            "input_dim": self.input_dim,
            "hidden_dim": self.hidden_dim,
            "output_dim": self.output_dim,
            "num_encode": self.num_encode,
            "num_decode": self.num_decode
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
import tensorflow as tf
from keras.layers import Input, Dense, Dropout, BatchNormalization, Embedding, Flatten, concatenate, MultiHeadAttention, LayerNormalization, Add
from keras.models import Model
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.metrics import AUC, Precision, Recall
import keras.backend as K


@tf.keras.utils.register_keras_serializable(package="Custom")
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerBlock, self).__init__(**kwargs)
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

    def get_config(self):
        config = super(TransformerBlock, self).get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "rate": self.rate
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

In [ ]:
print(tf.__version__)


In [ ]:
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]


In [ ]:
file_2021 = '/users/xwang259/scratch/NRD/NRD_2021_test.csv'
file_2022 = '/users/xwang259/scratch/NRD/NRD_2022_test.csv'

# Read and combine files
df1 = pd.read_csv(file_2021)
df2 = pd.read_csv(file_2022)

# Combine rows from both files
new_data = pd.concat([df1, df2], ignore_index=True)

# Convert all column names to uppercase
new_data.columns = new_data.columns.str.upper()

# Quick sanity check
print(new_data.shape)
print(new_data.head())
column_names = new_data.columns.tolist()
print(column_names)

In [ ]:
print(new_data['PAY1'])

In [ ]:
# One-hot encode 'PAY1' and 'ZIPINC_QRTL' only (excluding 'RACE')
new_data = pd.get_dummies(new_data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])

In [ ]:
# new_data_file_path = '/users/xwang259/icd/data/NRD_2019_Small.dta'
# new_data = pd.read_stata(new_data_file_path)

# new_data.columns = new_data.columns.str.upper()

# # Quick sanity check
# print(new_data.shape)
# print(new_data.head())
# column_names = new_data.columns.tolist()
# print(column_names)

In [ ]:
# Load the trained model
model = load_model('Model/mort_hypertrial_auc.keras')

# Load the LabelEncoder for ICD codes
with open('Model/full_label_encoder.pkl', 'rb') as file:
    encoder = pickle.load(file)

# Load the MinMaxScaler for 'AGE'
with open('Model/full_age_scaler.pkl', 'rb') as file:
    age_scaler = pickle.load(file)

In [ ]:
# # 2. Print a high-level summary (layer names, output shapes, parameter counts)
# model.summary()

# # 3. Get the full model config as a nested dict
# config = model.get_config()
# print(json.dumps(config, indent=2))

# # 4. If you want to see each layer’s hyperparameters one by one:
# for layer in model.layers:
#     print(f"Layer: {layer.name} ({layer.__class__.__name__})")
#     layer_config = layer.get_config()
#     for key, val in layer_config.items():
#         print(f"   {key}: {val}")
#     print()

In [ ]:

# Define the outcome variable and file path
outcome_var = 'MOR30'  # Set outcome variable here, e.g., 'DIED', 'MOR30', 'REA30'

# Filter out observations where DIED == 1 if outcome_var is REA30
# if outcome_var == 'REA30' and 'DIED' in new_data.columns:
if 'DIED' in new_data.columns:
    new_data = new_data[new_data['DIED'] != 1]

# Handle missing values in the target variable
new_data = new_data.dropna(subset=[outcome_var])

# Define ICD columns
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]

# Encode ICD codes
label_to_int = {label: idx for idx, label in enumerate(encoder.classes_)}
unknown_label_int = encoder.transform(["NAN"])[0]  # Assign unknown codes to index 0

for col in icd_columns:
    new_data[col] = new_data[col].astype(str).str.upper()  # Convert to uppercase
    new_data[col] = new_data[col].map(label_to_int).fillna(unknown_label_int).astype(int)

# Normalize 'AGE'
new_data['AGE'] = age_scaler.transform(new_data[['AGE']])

# One-hot encode 'PAY1' and 'ZIPINC_QRTL' only (excluding 'RACE')
new_data = pd.get_dummies(new_data, columns=['PAY1', 'ZIPINC_QRTL'], prefix=['PAY1', 'ZIPINC_QRTL'])

# Ensure that all expected one-hot encoded columns are present
def ensure_columns(data, expected_columns):
    for col in expected_columns:
        if col not in data.columns:
            data[col] = 0
    return data

# Define one-hot encoded columns for 'PAY1' and 'ZIPINC_QRTL' based on training data
pay1_columns = [col for col in new_data.columns if col.startswith('PAY1_')]
zipinc_qrtl_columns = [col for col in new_data.columns if col.startswith('ZIPINC_QRTL_')]
# Ensure all expected columns are present
new_data = ensure_columns(new_data, pay1_columns + zipinc_qrtl_columns)

In [ ]:
print(f"all payment column names: {pay1_columns}")
print(f"all zipinc quartile column names: {zipinc_qrtl_columns}")


In [ ]:
# Check for invalid PAY1 values
print("\n=== INSPECTING PAY1 VALUES ===")
print(f"Unique PAY1 values in new_data: {sorted(new_data['PAY1'].unique())}")
print(f"Value counts for PAY1:")
print(new_data['PAY1'].value_counts().sort_index())

In [ ]:

# Drop rows with missing values
X_new = new_data[['AGE', 'FEMALE'] + pay1_columns + zipinc_qrtl_columns + icd_columns]
X_new = X_new.dropna()
new_data = new_data.loc[X_new.index]

# Separate positive and negative rows
positives = new_data[new_data[outcome_var] == 1]
negatives = new_data[new_data[outcome_var] == 0]

# Sample 10% from each
sampled_positives = positives.sample(frac=0.1, random_state=42)
sampled_negatives = negatives.sample(frac=0.1, random_state=42)

# Combine the samples
combined_sampled_data = pd.concat([sampled_positives, sampled_negatives])

# Shuffle the combined data
new_data = shuffle(combined_sampled_data, random_state=42)

# Prepare input features. 
X_new = new_data[['AGE', 'FEMALE'] + pay1_columns + zipinc_qrtl_columns + icd_columns]
y_true = new_data[outcome_var].to_numpy(np.int32)

In [ ]:
# --- 1) one-shot predict with built-in batching (biggest win) ---
inputs = [
    X_new[icd_columns].to_numpy(np.float32),
    X_new['AGE'].to_numpy(np.float32),
    X_new['FEMALE'].to_numpy(np.float32),
] + [X_new[c].to_numpy(np.float32) for c in pay1_columns] \
  + [X_new[c].to_numpy(np.float32) for c in zipinc_qrtl_columns]

y_pred_prob = model.predict(inputs, batch_size=1024, verbose=0).squeeze()

In [ ]:
# Original AUC
original_auc = roc_auc_score(y_true, y_pred_prob)
print(f"AUC: {original_auc}")

In [ ]:
# import numpy as np
# import pandas as pd
# from sklearn.metrics import roc_auc_score, log_loss

# # ---------- helpers ----------
# def assemble_inputs(df, icd_array, pay1_cols, zip_cols):
#     """Build the input list exactly like your original predict call, but with a mutated ICD array."""
#     return [
#         icd_array.astype(np.float32),
#         df['AGE'].to_numpy(np.float32),
#         df['FEMALE'].to_numpy(np.float32),
#     ] + [df[c].to_numpy(np.float32) for c in pay1_cols] \
#       + [df[c].to_numpy(np.float32) for c in zip_cols]

# def batched_predict(model, inputs, batch_size=1024):
#     return model.predict(inputs, batch_size=batch_size, verbose=0).ravel()

# def stratified_sample_indices(y, sample_size, rng):
#     """Stratified by label to keep prevalence similar when subsampling."""
#     if sample_size is None or sample_size >= len(y):
#         return np.arange(len(y))
#     pos = np.where(y == 1)[0]
#     neg = np.where(y == 0)[0]
#     # maintain prevalence ratio
#     pos_target = max(1, int(round(sample_size * len(pos) / len(y))))
#     neg_target = sample_size - pos_target
#     pos_idx = rng.choice(pos, size=min(pos_target, len(pos)), replace=False)
#     neg_idx = rng.choice(neg, size=min(neg_target, len(neg)), replace=False)
#     idx = np.concatenate([pos_idx, neg_idx])
#     rng.shuffle(idx)
#     return np.sort(idx)

# def permute_code_once(X_codes, code_id, PAD_ID, rng):
#     """
#     Move `code_id` from random carrier rows to random non-carrier rows WITH PAD slots.
#     Keeps total count of code_id constant and avoids adding duplicates.
#     """
#     Xp = X_codes.copy()
#     carriers = np.unique(np.where(Xp == code_id)[0])
#     if carriers.size == 0:
#         return Xp, 0  # nothing to do

#     all_rows = np.arange(len(Xp))
#     noncar = np.setdiff1d(all_rows, carriers, assume_unique=True)

#     # receivers must (a) not carry the code, (b) have at least one PAD slot
#     has_pad = (Xp == PAD_ID).any(axis=1)
#     receivers = noncar[has_pad[noncar]]
#     if receivers.size == 0:
#         return Xp, 0  # nowhere safe to place the code

#     m = min(len(carriers), len(receivers))
#     src = rng.choice(carriers, size=m, replace=False)
#     dst = rng.choice(receivers, size=m, replace=False)

#     # perform the moves
#     for i, j in zip(src, dst):
#         # remove one occurrence from i (guaranteed to exist)
#         pos_i = np.where(Xp[i] == code_id)[0][0]
#         Xp[i, pos_i] = PAD_ID
#         # add into j at first PAD
#         pos_j = np.where(Xp[j] == PAD_ID)[0][0]
#         Xp[j, pos_j] = code_id
#     return Xp, m

# def permutation_importance_auc(
#     model,
#     df, y_true,
#     icd_cols, pay1_cols, zip_cols,
#     encoder,
#     PAD_LABEL="NAN",
#     K=5,
#     min_support=100,
#     batch_size=1024,
#     sample_size=None,
#     random_state=0,
# ):
#     """
#     Returns a DataFrame with per-code AUC drops and support, sorted by mean AUC drop desc.
#     """
#     rng = np.random.default_rng(random_state)

#     # Build core arrays (optionally subsample for speed)
#     X_icd_full = df[icd_cols].to_numpy(np.int32)
#     idx_eval = stratified_sample_indices(y_true, sample_size, rng)
#     y_eval = y_true[idx_eval]
#     X_icd = X_icd_full[idx_eval]

#     # Identify PAD/unknown id from the encoder
#     PAD_ID = encoder.transform([PAD_LABEL])[0]

#     # Base predictions / AUC
#     base_pred = batched_predict(
#         model, assemble_inputs(df.iloc[idx_eval], X_icd, pay1_cols, zip_cols),
#         batch_size=batch_size
#     )
#     base_auc = roc_auc_score(y_eval, base_pred)
#     base_logloss = log_loss(y_eval, base_pred)

#     # Candidate codes: appear at least min_support times in the EVAL split
#     unique_codes, counts = np.unique(X_icd, return_counts=True)
#     mask = (unique_codes != PAD_ID) & (counts >= min_support)
#     cand_codes = unique_codes[mask]
#     cand_support = {int(cid): int(cnt) for cid, cnt in zip(unique_codes[mask], counts[mask])}

#     results = []

#     for code_id in tqdm(cand_codes):
#         auc_drops = []
#         logloss_drops = []
#         for _ in range(K):
#             Xp, moved = permute_code_once(X_icd, int(code_id), PAD_ID, rng)
#             if moved == 0:
#                 auc_drops.append(0.0)
#                 logloss_drops.append(0.0)
#                 continue
#             pred_perm = batched_predict(
#                 model, assemble_inputs(df.iloc[idx_eval], Xp, pay1_cols, zip_cols),
#                 batch_size=batch_size
#             )
#             auc_perm = roc_auc_score(y_eval, pred_perm)
#             logloss_perm = log_loss(y_eval, pred_perm)
#             auc_drops.append(base_auc - auc_perm)
#             logloss_drops.append(logloss_perm - base_logloss)
#         results.append({
#             "code_id": int(code_id),
#             "icd_code": encoder.inverse_transform([int(code_id)])[0],
#             "support_eval": cand_support[int(code_id)],
#             "mean_auc_drop": float(np.mean(auc_drops)),
#             "sd_auc_drop": float(np.std(auc_drops)),
#             "mean_logloss_drop": float(np.mean(logloss_drops)),
#             "sd_logloss_drop": float(np.std(logloss_drops)),
#             "K": K
#         })
        

#     res = pd.DataFrame(results).sort_values("mean_auc_drop", ascending=False)

#     # Decode code ids back to ICD strings
#     if len(res) > 0:
#         res["icd_code"] = encoder.inverse_transform(res["code_id"].to_numpy())
#         cols = ["icd_code", "code_id", "support_eval", "mean_auc_drop", "sd_auc_drop", "K"]
#         res = res[cols]

#     return res, base_auc, base_logloss

# # ---------- run it ----------
# # Compute permutation importance
# perm_df, base_auc, base_logloss = permutation_importance_auc(
#     model=model,
#     df=new_data,
#     y_true=y_true,
#     icd_cols=icd_columns,
#     pay1_cols=pay1_columns,
#     zip_cols=zipinc_qrtl_columns,
#     encoder=encoder,
#     PAD_LABEL="NAN",     # this is what you mapped empties/unknowns to
#     K=5,                 # increase for more stable estimates
#     min_support=200,     # tweak based on your dataset size
#     batch_size=1024,
#     sample_size=None,    # e.g., 100_000 for speed; None uses all rows
#     random_state=0,
# )
# print(f"Base AUC: {base_auc:.6f}, Base LogLoss: {base_logloss:.6f}")
# print(perm_df.head(10).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

# ---------- config ----------
STEPS = 32               # IG Riemann steps (try 32–64)
BATCH_SIZE = 1024
SAMPLE_SIZE = None       # e.g., 50_000 for speed; None = all rows
PAD_LABEL = "NAN"        # your PAD/unknown label
TARGET = "logit"         # "logit" or "loss"
ICD_EMBED_VAR_NAME = None  # set to exact var name if auto-detect fails

# ---------- utilities reused from before ----------
def assemble_inputs(df, icd_array, pay1_cols, zip_cols):
    return [
        icd_array.astype(np.float32),
        df['AGE'].to_numpy(np.float32),
        df['FEMALE'].to_numpy(np.float32),
    ] + [df[c].to_numpy(np.float32) for c in pay1_cols] \
      + [df[c].to_numpy(np.float32) for c in zip_cols]

def stratified_sample_indices(y, sample_size, rng):
    if sample_size is None or sample_size >= len(y):
        return np.arange(len(y))
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    pos_target = max(1, int(round(sample_size * len(pos) / len(y))))
    neg_target = sample_size - pos_target
    pos_idx = rng.choice(pos, size=min(pos_target, len(pos)), replace=False)
    neg_idx = rng.choice(neg, size=min(neg_target, len(neg)), replace=False)
    idx = np.concatenate([pos_idx, neg_idx])
    rng.shuffle(idx)
    return np.sort(idx)

def find_icd_embedding_var(model, vocab_size_hint=None, name_hint=None):
    candidates = []
    for v in model.trainable_variables:
        if v.shape.rank == 2 and "emb" in v.name.lower():
            candidates.append((v.name, tuple(v.shape.as_list())))
    if candidates:
        print("Embedding-like variables:")
        for n, s in candidates:
            print(f"  - {n} shape={s}")
    if name_hint is not None:
        for v in model.trainable_variables:
            if v.name == name_hint:
                return v
    if vocab_size_hint is not None:
        best = None
        for v in model.trainable_variables:
            if v.shape.rank == 2 and v.shape[0] >= vocab_size_hint:
                if ("emb" in v.name.lower()) or best is None:
                    best = v
        if best is not None:
            return best
    two_d_embs = [v for v in model.trainable_variables if v.shape.rank == 2 and "emb" in v.name.lower()]
    if two_d_embs:
        return max(two_d_embs, key=lambda t: int(t.shape[0]) * int(t.shape[1]))
    raise ValueError("Could not identify ICD embedding variable. Set ICD_EMBED_VAR_NAME explicitly.")

# ---------- main IG function ----------
def integrated_gradients_icd_global(
    model, df, y_true, icd_cols, pay1_cols, zip_cols, encoder,
    steps=STEPS, batch_size=BATCH_SIZE, sample_size=SAMPLE_SIZE,
    pad_label=PAD_LABEL, target=TARGET, baseline="pad",  # "pad" | "zero" | "mean"
    icd_embed_var_name=ICD_EMBED_VAR_NAME, random_state=0,
):
    rng = np.random.default_rng(random_state)
    # subset (optional)
    X_icd_full = df[icd_cols].to_numpy(np.int32)
    idx_eval = stratified_sample_indices(y_true, sample_size, rng)
    df_eval = df.iloc[idx_eval]
    y_eval = y_true[idx_eval].astype(np.float32)
    X_icd_eval = X_icd_full[idx_eval]

    # counts per code for per-occurrence normalization
    vocab_size_hint = len(encoder.classes_)
    counts = np.bincount(X_icd_eval.reshape(-1), minlength=vocab_size_hint)

    # embedding var & PAD
    emb_var = find_icd_embedding_var(model, vocab_size_hint=vocab_size_hint, name_hint=icd_embed_var_name)
    E_orig = emb_var.read_value()                # (V, D)
    PAD_ID = int(encoder.transform([pad_label])[0])

    # build baseline matrix E0
    if baseline == "pad":
        e_pad = tf.gather(E_orig, [PAD_ID])[0]   # (D,)
        E0 = tf.repeat(e_pad[None, :], repeats=int(E_orig.shape[0]), axis=0)
    elif baseline == "zero":
        E0 = tf.zeros_like(E_orig)
    elif baseline == "mean":
        E0 = tf.repeat(tf.reduce_mean(E_orig, axis=0, keepdims=True), repeats=int(E_orig.shape[0]), axis=0)
    else:
        raise ValueError("baseline must be one of {'pad','zero','mean'}")

    deltaE = E_orig - E0                         # (V, D)

    # data pipeline
    inputs_eval = assemble_inputs(df_eval, X_icd_eval, pay1_cols, zip_cols)
    ds = tf.data.Dataset.from_tensor_slices((tuple(tf.convert_to_tensor(a) for a in inputs_eval),
                                             tf.convert_to_tensor(y_eval))).batch(batch_size)

    # accumulators
    grads_accum = tf.zeros_like(E_orig)          # sum over steps & batches
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)

    # IG loop over steps
    try:
        for s in tqdm(range(1, steps + 1)):
            alpha = tf.cast(s / steps, tf.float32)
            E_interp = E0 + alpha * deltaE       # (V, D)
            emb_var.assign(E_interp)             # temporarily set embeddings

            for (batch_inputs, yb) in ds:
                with tf.GradientTape() as tape:
                    tape.watch(emb_var)
                    out = model(batch_inputs, training=False)
                    out = tf.reshape(out, (-1,))   # (B,)
                    if target == "logit":
                        eps = tf.constant(1e-6, out.dtype)
                        logit = tf.math.log(tf.clip_by_value(out, eps, 1.0 - eps)) \
                                - tf.math.log1p(-tf.clip_by_value(out, eps, 1.0 - eps))
                        scalar = tf.reduce_mean(logit)
                    elif target == "loss":
                        loss = bce(yb, out)
                        scalar = tf.reduce_mean(loss)
                    else:
                        raise ValueError("target must be 'logit' or 'loss'")
                g = tape.gradient(scalar, emb_var)   # (V, D); non-zero only for codes present in batch
                grads_accum += g

        grads_avg = grads_accum / steps              # approximate integral average
        # per-code IG: sum over D of (deltaE * grads_avg)
        per_code_signed = tf.reduce_sum(deltaE * grads_avg, axis=1)  # (V,)
        per_code_mag = tf.math.abs(per_code_signed)

        # build results
        codes = np.arange(per_code_mag.shape[0])
        mask_valid = (codes != PAD_ID) & (counts > 0)
        codes = codes[mask_valid]
        occ = counts[mask_valid]
        ig_total = per_code_mag.numpy()[mask_valid]
        ig_mean = ig_total / np.maximum(occ, 1)

        icd_str = encoder.inverse_transform(codes)
        res = pd.DataFrame({
            "icd_code": icd_str,
            "code_id": codes,
            "support_eval": occ,
            "ig_total_mag": ig_total,
            "ig_mean_per_occurrence": ig_mean,
            "target": target,
            "baseline": baseline,
            "steps": steps,
        })
        # two views
        top50_global = res.sort_values("ig_total_mag", ascending=False).head(50)
        top50_perocc = res[res["support_eval"] >= 50].sort_values("ig_mean_per_occurrence", ascending=False).head(50)

        return res, top50_global, top50_perocc
    finally:
        # restore original embeddings no matter what
        emb_var.assign(E_orig)

In [ ]:
res_ig, top50_global_ig, top50_perocc_ig = integrated_gradients_icd_global(
    model=model,
    df=new_data,
    y_true=y_true.astype(np.float32),
    icd_cols=icd_columns,
    pay1_cols=pay1_columns,
    zip_cols=zipinc_qrtl_columns,
    encoder=encoder,
    steps=32,                # try 64 if you want tighter completeness
    batch_size=1024,
    sample_size=None,        # e.g., 100_000 for speed
    pad_label="NAN",
    target="logit",          # or "loss"
    baseline="pad",          # "pad" | "zero" | "mean"
)
print(top50_global_ig.head(10).to_string(index=False))
print(top50_perocc_ig.head(10).to_string(index=False))


In [ ]:
# ---- Fast global ICD importance via Grad × Embedding (Keras/TF) ----
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils import resample

# CONFIG
BATCH_SIZE = 1024
SAMPLE_SIZE = None     # e.g., 50_000 for speed; None = use all rows
PAD_LABEL = "NAN"      # your PAD/unknown label
ICD_INPUT_INDEX = 0    # your ICD token array is first in `inputs` list
ICD_EMBED_VAR_NAME = None  # e.g., "icd_embedding/embeddings:0" if you know it

# Build inputs exactly like your predict() call
def assemble_inputs(df, icd_array, pay1_cols, zip_cols):
    return [
        icd_array.astype(np.float32),
        df['AGE'].to_numpy(np.float32),
        df['FEMALE'].to_numpy(np.float32),
    ] + [df[c].to_numpy(np.float32) for c in pay1_cols] \
      + [df[c].to_numpy(np.float32) for c in zip_cols]

def stratified_sample_indices(y, sample_size, rng):
    if sample_size is None or sample_size >= len(y):
        return np.arange(len(y))
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    pos_target = max(1, int(round(sample_size * len(pos) / len(y))))
    neg_target = sample_size - pos_target
    pos_idx = rng.choice(pos, size=min(pos_target, len(pos)), replace=False)
    neg_idx = rng.choice(neg, size=min(neg_target, len(neg)), replace=False)
    idx = np.concatenate([pos_idx, neg_idx])
    rng.shuffle(idx)
    return np.sort(idx)

def find_icd_embedding_var(model, vocab_size_hint=None, name_hint=None):
    # Print candidates to help you pick the right variable if needed
    candidates = []
    for v in model.trainable_variables:
        if v.shape.rank == 2 and "emb" in v.name.lower():  # heuristic
            candidates.append((v.name, tuple(v.shape.as_list())))
    if candidates:
        print("Embedding-like variables:")
        for n, s in candidates:
            print(f"  - {n} shape={s}")
    # Choose by explicit name first
    if name_hint is not None:
        for v in model.trainable_variables:
            if v.name == name_hint:
                return v
    # Else choose by vocab size hint
    if vocab_size_hint is not None:
        best = None
        for v in model.trainable_variables:
            if v.shape.rank == 2 and v.shape[0] >= vocab_size_hint:
                if ("emb" in v.name.lower()) or best is None:
                    best = v
        if best is not None:
            return best
    # Fallback: largest 2D var containing "emb"
    two_d_embs = [v for v in model.trainable_variables if v.shape.rank == 2 and "emb" in v.name.lower()]
    if two_d_embs:
        return max(two_d_embs, key=lambda t: int(t.shape[0]) * int(t.shape[1]))
    raise ValueError("Could not identify ICD embedding variable. Set ICD_EMBED_VAR_NAME explicitly.")

# Prep data subset
rng = np.random.default_rng(0)
icd_columns = [f'I10_DX{i}' for i in range(1, 41)]
pay1_columns = [c for c in new_data.columns if c.startswith('PAY1_')]
zipinc_qrtl_columns = [c for c in new_data.columns if c.startswith('ZIPINC_QRTL_')]

X_icd_full = new_data[icd_columns].to_numpy(np.int32)
idx_eval = stratified_sample_indices(y_true, SAMPLE_SIZE, rng)
df_eval = new_data.iloc[idx_eval]
y_eval = y_true[idx_eval]
X_icd_eval = X_icd_full[idx_eval]
PAD_ID = encoder.transform([PAD_LABEL])[0]

# Count occurrences per code in the eval split (for per-occurrence normalization)
vocab_size_hint = len(encoder.classes_)
counts = np.bincount(X_icd_eval.reshape(-1), minlength=vocab_size_hint)

# Find the ICD embedding variable
emb_var = find_icd_embedding_var(model, vocab_size_hint=vocab_size_hint, name_hint=ICD_EMBED_VAR_NAME)
print(f"Using embedding variable: {emb_var.name}, shape={emb_var.shape}")

In [ ]:
# Build a tf.data pipeline for speed
inputs_eval = assemble_inputs(df_eval, X_icd_eval, pay1_columns, zipinc_qrtl_columns)
# Convert lists of arrays to tf.data Dataset of tuples
dataset = tf.data.Dataset.from_tensor_slices((
    tuple(tf.convert_to_tensor(arr) for arr in inputs_eval),
    tf.convert_to_tensor(y_eval, dtype=tf.float32)
)).batch(BATCH_SIZE)

# Accumulate Grad×Embedding attribution per code across all batches
attr_sum = tf.zeros((emb_var.shape[0],), dtype=tf.float32)

bce = tf.keras.losses.BinaryCrossentropy(from_logits=False, reduction=tf.keras.losses.Reduction.NONE)

for (batch_inputs, yb) in tqdm(dataset):
    with tf.GradientTape() as tape:
        tape.watch(emb_var)  # watch the embedding weights
        preds = model(batch_inputs, training=False)  # shape (B, 1) or (B,)
        preds = tf.reshape(preds, (-1,))
        # label-aware: gradients of **loss** (more sensitive than prob alone)
        loss = bce(yb, preds)
        loss = tf.reduce_mean(loss)

    grads = tape.gradient(loss, emb_var)               # shape (V, D)
    # Grad × Input over the embedding rows
    # gxi = tf.math.abs(grads * emb_var)                # (V, D)
    gxi = grads * emb_var                               # (V, D)
    per_code = tf.reduce_sum(gxi, axis=1)             # (V,)
    attr_sum += per_code

attr_sum = attr_sum.numpy()

# Build results table
codes = np.arange(len(attr_sum))
mask_valid = (codes != PAD_ID) & (counts > 0)
codes = codes[mask_valid]
tot_attr = attr_sum[mask_valid]
occ = counts[mask_valid]
mean_attr = tot_attr / np.maximum(occ, 1)

icd_str = encoder.inverse_transform(codes)

res = pd.DataFrame({
    "icd_code": icd_str,
    "code_id": codes,
    "support_eval": occ,
    "total_attr_gxi": tot_attr,
    "mean_attr_per_occurrence": mean_attr
})

min_support = 0
top_num = 20

# Two views: "global impact" vs "per-occurrence effect"
top20_global = res.sort_values("total_attr_gxi", ascending=False).head(top_num)
top20_per_occ = res[res["support_eval"] >= min_support].sort_values("mean_attr_per_occurrence", ascending=False).head(top_num)

print(f"\nTop {top_num} ICDs by GLOBAL impact (Grad×Embedding, label-aware):")
print(top20_global[["icd_code","support_eval","total_attr_gxi","mean_attr_per_occurrence"]].to_string(index=False))

print(f"\nTop {top_num} ICDs by PER-OCCURRENCE effect (min support={min_support}):")
print(top20_per_occ[["icd_code","support_eval","mean_attr_per_occurrence","total_attr_gxi"]].to_string(index=False))

In [ ]:

min_support = 50
top_num = 20

# Two views: "global impact" vs "per-occurrence effect"
top20_global = res.sort_values("total_attr_gxi", ascending=False).head(top_num)
top20_per_occ = res[res["support_eval"] >= min_support].sort_values("mean_attr_per_occurrence", ascending=False).head(top_num)

print(f"\nTop {top_num} ICDs by GLOBAL impact (Grad×Embedding, label-aware):")
print(top20_global[["icd_code","support_eval","total_attr_gxi","mean_attr_per_occurrence"]].to_string(index=False))

print(f"\nTop {top_num} ICDs by PER-OCCURRENCE effect (min support={min_support}):")
print(top20_per_occ[["icd_code","support_eval","mean_attr_per_occurrence","total_attr_gxi"]].to_string(index=False))

In [ ]:
# --- 2) AUC computed once + fast 95% CI (Hanley–McNeil) ---
def auc_ci_hm(y, p, ci=0.95):
    y = np.asarray(y, dtype=np.int32)
    p = np.asarray(p, dtype=np.float64)
    auc = roc_auc_score(y, p)
    n1 = (y == 1).sum()
    n0 = (y == 0).sum()
    Q1 = auc / (2 - auc)
    Q2 = 2 * auc**2 / (1 + auc)
    var = (auc*(1-auc) + (n1-1)*(Q1 - auc**2) + (n0-1)*(Q2 - auc**2)) / (n1*n0)
    se = np.sqrt(max(var, 0.0))
    # use z=1.96 for 95% without scipy
    z = 1.96
    return max(0.0, auc - z*se), min(1.0, auc + z*se)

In [ ]:
auc = roc_auc_score(y_true, y_pred_prob)
auc_lower, auc_upper = auc_ci_hm(y_true, y_pred_prob)

# --- 3) vectorized metrics for all thresholds at once ---
thresholds = np.arange(0.1, 1.0, 0.1)
pred_mat = (y_pred_prob[:, None] > thresholds[None, :])  # shape [N, T]

pos = (y_true == 1)[:, None]
neg = ~pos

TP = np.sum(pred_mat & pos, axis=0).astype(np.int64)
FP = np.sum(pred_mat & neg, axis=0).astype(np.int64)
FN = np.sum((~pred_mat) & pos, axis=0).astype(np.int64)
TN = np.sum((~pred_mat) & neg, axis=0).astype(np.int64)

precision = np.divide(TP, TP + FP, out=np.zeros_like(TP, dtype=float), where=(TP+FP) > 0)
recall    = np.divide(TP, TP + FN, out=np.zeros_like(TP, dtype=float), where=(TP+FN) > 0)
accuracy  = (TP + TN) / (TP + TN + FP + FN)

f1 = np.divide(2*precision*recall, precision + recall,
               out=np.zeros_like(precision), where=(precision+recall) > 0)

beta2 = 4.0  # (beta=2)^2
f2 = np.divide((1+beta2)*precision*recall, beta2*precision + recall,
               out=np.zeros_like(precision), where=(beta2*precision + recall) > 0)

results_df = pd.DataFrame({
    'Threshold': np.round(thresholds, 1),
    'AUC': auc,                     # same for all rows
    'AUC_CI_Lower': auc_lower,
    'AUC_CI_Upper': auc_upper,
    'Accuracy': np.round(accuracy, 4),
    'Precision': np.round(precision, 4),
    'Recall': np.round(recall, 4),
    'F1 Score': np.round(f1, 4),
    'F2 Score': np.round(f2, 4),
})

print("\nModel Performance Metrics Across Different Thresholds:")
print(results_df.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score

# ---------- helpers ----------
def auc_ci_hm(y, p, ci=0.95):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=np.float64)
    A = roc_auc_score(y, p)
    n1 = int((y == 1).sum())
    n0 = int((y == 0).sum())
    Q1 = A / (2 - A)
    Q2 = 2 * A * A / (1 + A)
    var = (A*(1-A) + (n1-1)*(Q1 - A*A) + (n0-1)*(Q2 - A*A)) / (n1*n0)
    se = np.sqrt(max(var, 0.0))
    z = 1.96  # 95%
    return max(0.0, A - z*se), min(1.0, A + z*se)

def counts_from_preds(y_true, y_pred):
    y_true = y_true.astype(np.int8)
    y_pred = y_pred.astype(np.int8)
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, FP, FN, TN

def metrics_from_counts(TP, FP, FN, TN):
    denom = TP + FP + FN + TN
    accuracy  = (TP + TN) / denom if denom else 0.0
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = (2*precision*recall) / (precision + recall) if (precision + recall) else 0.0
    beta2 = 4.0  # beta=2
    f2 = ((1+beta2)*precision*recall) / (beta2*precision + recall) if (beta2*precision + recall) else 0.0
    return accuracy, precision, recall, f1, f2

# =========================
# 1) FAST Youden threshold
# =========================
if len(np.unique(y_true)) > 1:
    y = np.asarray(y_true, dtype=np.int8)
    p = np.asarray(y_pred_prob, dtype=np.float64)

    # Exact Youden optimum from ROC (Youden = TPR - FPR)
    fpr, tpr, thr = roc_curve(y, p)
    mask = np.isfinite(thr)         # drop the initial 'inf' threshold
    youden = tpr[mask] - fpr[mask]  # vectorized
    i = int(np.argmax(youden))

    best_threshold = float(thr[mask][i])
    best_value     = float(youden[i])
    sensitivity    = float(tpr[mask][i])
    specificity    = float(1.0 - fpr[mask][i])

    print(f"Best Youden index threshold: {best_threshold}")
    print(f"Best Youden index value: {best_value:.6f}")
    print(f"Sensitivity at best: {sensitivity:.4f}")
    print(f"Specificity at best: {specificity:.4f}")

    # Metrics at optimal threshold (single pass)
    y_pred_optimal = (p > best_threshold).astype(np.int8)
    TP, FP, FN, TN = counts_from_preds(y, y_pred_optimal)
    acc, prec, rec, f1, f2 = metrics_from_counts(TP, FP, FN, TN)

    auc = roc_auc_score(y, p)
    auc_lower, auc_upper = auc_ci_hm(y, p)

    print(f"AUC: {auc:.4f}")
    print(f"95% CI for AUC: [{auc_lower:.4f}, {auc_upper:.4f}]")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"F2 Score: {f2:.4f}")

# =============================================
# 2) FAST validation of traditional score indexes
# =============================================
traditional_indexes = ['INDEX_MORTALITY', 'INDEX_READMISSION', 'CHARLINDEX', 'CHARLINDEX_AGE_ADJUST']
results = []

for index in traditional_indexes:
    print(f"Validating {index} against {outcome_var}...")

    # Drop rows with NA in either column (actually do it, not just comment)
    data = new_data[[index, outcome_var]].dropna()
    y_true = data[outcome_var].to_numpy(np.int8)
    scores = data[index].to_numpy(np.float64)

    if len(np.unique(y_true)) < 2:
        print(f"Skipped {index}: Only one class present in {outcome_var}")
        continue

    # AUC + fast CI (single computation each)
    auc = roc_auc_score(y_true, scores)
    auc_lower, auc_upper = auc_ci_hm(y_true, scores)

    # Binary prediction at > 0 (keeps your original behavior)
    y_pred = (scores > 0).astype(np.int8)
    TP, FP, FN, TN = counts_from_preds(y_true, y_pred)
    acc, prec, rec, f1, f2 = metrics_from_counts(TP, FP, FN, TN)

    results.append({
        'Index': index,
        'AUC': auc,
        'AUC Lower CI': auc_lower,
        'AUC Upper CI': auc_upper,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'F2 Score': f2
    })

if results:
    results_df = pd.DataFrame(results).round(4)
    print("\nValidation Metrics for Traditional Scores:")
    print(results_df.to_string(index=False))
else:
    print("No metrics were calculated due to insufficient class diversity in the data.")


In [ ]:
y_new = y_true

# Make predictions
batch_size = 1024
num_samples = len(X_new)
y_pred_prob = []

print(f'length of test data: {num_samples}')

# Prepare the inputs in the same structure as during training
# for start_idx in tqdm(range(0, num_samples, batch_size), desc="Making Predictions"):
for start_idx in tqdm(range(0, num_samples, batch_size)):

    end_idx = min(start_idx + batch_size, num_samples)
    batch_inputs = [
        X_new[icd_columns].iloc[start_idx:end_idx],
        X_new['AGE'].iloc[start_idx:end_idx].values,
        X_new['FEMALE'].iloc[start_idx:end_idx].values,
    ] + [X_new[col].iloc[start_idx:end_idx].values for col in pay1_columns] \
      + [X_new[col].iloc[start_idx:end_idx].values for col in zipinc_qrtl_columns]

    batch_pred_prob = model.predict(batch_inputs, verbose=0)
    y_pred_prob.extend(batch_pred_prob.flatten())

y_pred_prob = np.array(y_pred_prob)

# Function to calculate the 95% CI for AUC using bootstrapping
def calculate_auc_ci(y_true, y_pred_prob, n_bootstraps=1000, ci=0.95):
    bootstrapped_aucs = []
    np.random.seed(42)

    for i in range(n_bootstraps):
        # Sample with replacement from the data
        indices = np.random.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[indices])) < 2:
            # Skip resamples that do not have both classes present
            continue

        score = roc_auc_score(y_true[indices], y_pred_prob[indices])
        bootstrapped_aucs.append(score)

    # Calculate CI
    sorted_aucs = np.sort(bootstrapped_aucs)
    lower_bound = sorted_aucs[int((1 - ci) / 2 * len(sorted_aucs))]
    upper_bound = sorted_aucs[int((1 + ci) / 2 * len(sorted_aucs))]
    return lower_bound, upper_bound


# Evaluate the model if true labels are available
if y_new is not None:
    thresholds = np.arange(0.1, 1.0, 0.1)
    results = []
    
    for threshold in thresholds:
        y_pred = (y_pred_prob > threshold).astype(int)
        
        if len(np.unique(y_new)) > 1:
            auc = roc_auc_score(y_new, y_pred_prob)
            auc_lower, auc_upper = calculate_auc_ci(y_new, y_pred_prob)
            accuracy = accuracy_score(y_new, y_pred)
            precision = precision_score(y_new, y_pred, zero_division=0)
            recall = recall_score(y_new, y_pred, zero_division=0)
            f1 = f1_score(y_new, y_pred, zero_division=0)
            f2 = f2_score(y_new, y_pred)
            
            results.append({
                'Threshold': round(threshold, 1),
                'AUC': auc,
                'AUC_CI_Lower': auc_lower,
                'AUC_CI_Upper': auc_upper,
                'Accuracy': accuracy,
                'Precision': precision,
                'Recall': recall,
                'F1 Score': f1,
                'F2 Score': f2
            })
        else:
            accuracy = accuracy_score(y_new, y_pred)
            results.append({
                'Threshold': round(threshold, 1),
                'Accuracy': accuracy
            })
    
    # Create and display results table
    results_df = pd.DataFrame(results)
    if len(np.unique(y_new)) > 1:
        results_df = results_df.round(4)
        results_df = results_df[['Threshold', 'AUC', 'AUC_CI_Lower', 'AUC_CI_Upper', 
                               'Accuracy', 'Precision', 'Recall', 'F1 Score', 'F2 Score']]
    else:
        results_df = results_df.round(4)
        results_df = results_df[['Threshold', 'Accuracy']]
    
    print("\nModel Performance Metrics Across Different Thresholds:")
    print(results_df.to_string(index=False))
else:
    print("Predictions:")
    print(y_pred_prob)

In [ ]:

# Step 2: Find optimal threshold maximizing Youden's Index
if len(np.unique(y_new)) > 1:
    # Use a finer range of thresholds (0 to 1 with 0.01 increments)
    fine_thresholds = np.arange(0.0, 1.01, 0.01)
    youden_results = []
    
    for threshold in tqdm(fine_thresholds):
        y_pred = (y_pred_prob > threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_new, y_pred).ravel()
        sensitivity = recall_score(y_new, y_pred, zero_division=0)
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        youdens_index = sensitivity + specificity - 1
        
        youden_results.append({
            'Threshold': threshold,
            'Sensitivity': sensitivity,
            'Specificity': specificity,
            'Youden\'s Index': youdens_index
        })
    
    # Create DataFrame and find optimal threshold
    youden_df = pd.DataFrame(youden_results)
    optimal_idx = youden_df['Youden\'s Index'].idxmax()
    optimal_threshold = youden_df.loc[optimal_idx]
    
    best_threshold = optimal_threshold['Threshold']
    best_value = optimal_threshold['Youden\'s Index']
    print(f"Best Youden index threshold: {best_threshold}")
    print(f"Best Youden index value: {best_value}")

    # Calculate metrics at optimal threshold
    y_pred_optimal = (y_pred_prob > optimal_threshold['Threshold']).astype(int)
    

    auc = roc_auc_score(y_new, y_pred_prob)
    auc_lower, auc_upper = calculate_auc_ci(y_new, y_pred_prob)
    accuracy = accuracy_score(y_new, y_pred_optimal)
    precision = precision_score(y_new, y_pred_optimal, zero_division=0)
    recall = recall_score(y_new, y_pred_optimal, zero_division=0)
    f1 = f1_score(y_new, y_pred_optimal, zero_division=0)
    f2 = f2_score(y_new, y_pred_optimal)

    print(f"AUC: {auc:.4f}")
    print(f"95% CI for AUC: [{auc_lower:.4f}, {auc_upper:.4f}]")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"F2 Score: {f2:.4f}")

In [ ]:


# List of traditional indexes to validate
traditional_indexes = ['INDEX_MORTALITY', 'INDEX_READMISSION', 'CHARLINDEX', 'CHARLINDEX_AGE_ADJUST']

results = []

# Validate each traditional index
for index in traditional_indexes:
    print(f"Validating {index} against {outcome_var}...")

    
    # Filter data to exclude rows with missing values in the index or outcome variable
    data = new_data[[index, outcome_var]]
    y_true = data[outcome_var].to_numpy()
    y_pred_prob = data[index].to_numpy()

    # Convert index scores to binary predictions (if applicable)
    y_pred = (y_pred_prob > 0).astype(int)

    # Ensure both classes are present
    if len(np.unique(y_true)) > 1:
        # Calculate evaluation metrics
        auc = roc_auc_score(y_true, y_pred_prob)
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        f2 = f2_score(y_true, y_pred)

        # Calculate AUC confidence intervals
        auc_lower, auc_upper = calculate_auc_ci(y_true, y_pred_prob)

        # Store results
        results.append({
            'Index': index,
            'AUC': auc,
            'AUC Lower CI': auc_lower,
            'AUC Upper CI': auc_upper,
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'F2 Score': float(f2)
        })
    else:
        print(f"Skipped {index}: Only one class present in {outcome_var}")

# Convert results to a DataFrame for easier analysis
if results:
    results_df = pd.DataFrame(results)
    print("\nValidation Metrics for Traditional Scores:")
    print(results_df.to_string(index=False))  # Clean output without indices
else:
    print("No metrics were calculated due to insufficient class diversity in the data.")